In [2]:
import os
os.chdir('..')

In [3]:
%pwd

'e:\\MLOPS\\bid_bot_detection'

In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Import

In [ ]:
bids = pd.read_csv('/raw/data/bids.csv', sep=',')
print('bids: OK {}'.format(bids.shape))
bids.head()
# bids = pd.read_csv('data/bids_normalized.csv', sep=',')
# print('bids: OK {}'.format(bids.shape))
# bids.head()

In [ ]:
b_train = pd.read_csv('../data/train.csv', sep=',')
print('b_train: OK: {}'.format(b_train.shape))
b_train.head()
# b_train = pd.read_csv('data/b_train.csv', sep=',')
# print('b_train: OK: {}'.format(b_train.shape))
# b_train.head()

In [ ]:
b_test = pd.read_csv('../data/test.csv', sep=',')
print('b_test: OK: {}'.format(b_test.shape))
b_test.head()
# b_test = pd.read_csv('data/b_test.csv', sep=',')
# print('b_test: OK: {}'.format(b_test.shape))
# b_test.head()

# Preprocessing

## bids_normalized table

In [ ]:
bids.head()

### 1. time normalized

In [ ]:
min_time = bids['time'].min()
bids['time_normalized'] = round((bids['time'] - min_time) * 19 / 1e9,2)
pd.DataFrame(bids.time_normalized.unique())
bids.head()

In [ ]:
min_time = bids['time'].min()
bids['time_normalized'] = round((bids['time'] - min_time) * 19/1e9, 2)

In [ ]:
pd.DataFrame(bids.time_normalized.unique()) # shows that we succesfully increment the time 1 at a time
bids.head()

### 2. outcome

In [ ]:
bids['outcome'] = bids.merge(b_train, how='left', on='bidder_id').outcome

### other

In [ ]:
del bids['merchandise']
del bids['device']
del bids['country']
del bids['ip']
del bids['url']
bids.to_csv('../data/bids_normalized_short.csv', sep=',', header=True, index=False)

In [ ]:
bids

## features for b_train & b_test table

In [ ]:
del b_train['address']
del b_train['payment_account']

del b_test['address']
del b_test['payment_account']

In [ ]:
b_train

### 1. mean_time_diff

In [ ]:
b_train['mean_time_diff'] = 0
for bidder in b_train['bidder_id']:
    #d = bids[bids.bidder_id == bidder].sort_values('time_normalized')[['time_normalized']] - bids[bids.bidder_id == bidder].sort_values('time_normalized')[['time_normalized']].shift(1)
    #b_train.loc[b_train[b_train.bidder_id == bidder].index,'mean_time_diff'] = (bids[bids.bidder_id == bidder].sort_values('time_normalized')[['time_normalized']] - bids[bids.bidder_id == bidder].sort_values('time_normalized')[['time_normalized']].shift(1)).mean()

    m = (bids[bids.bidder_id == bidder].sort_values('time_normalized')[['time_normalized']] -
         bids[bids.bidder_id == bidder].sort_values('time_normalized')[['time_normalized']].shift(1)).mean()[0]
    b_train.loc[b_train[b_train.bidder_id == bidder].index, 'mean_time_diff'] = m
    # b_train.to_csv('data/b_train.csv', sep=',', header=True, index=False)
b_train.loc[:,'mean_time_diff'] = round(b_train.mean_time_diff, 4)

In [ ]:
b_train

In [ ]:
b_test['mean_time_diff'] = 0
i = 0
for bidder in b_test['bidder_id']:
    if i % 200 == 0:
        print(i)
    i += 1
    m = (bids[bids.bidder_id == bidder].sort_values('time_normalized')[['time_normalized']] -
         bids[bids.bidder_id == bidder].sort_values('time_normalized')[['time_normalized']].shift(1)).mean()[0]
    b_test.loc[b_test[b_test.bidder_id == bidder].index, 'mean_time_diff'] = m
    # b_test.to_csv('data/b_test.csv', sep=',', header=True, index=False)
b_test.loc[:,'mean_time_diff'] = round(b_test.mean_time_diff, 4)

### 2. total_bids

In [ ]:
b_train['total_bids'] = 0
i = 0
for bidder in b_train['bidder_id']:
    if i % 100 == 0:
        print(i)
    i += 1
    c = bids[bids.bidder_id == bidder].count()[0]
    b_train.loc[b_train[b_train.bidder_id == bidder].index, 'total_bids'] = c
    # b_train.to_csv('data/b_train.csv', sep=',', header=True, index=False)

In [ ]:
b_test['total_bids'] = 0
i = 0
for bidder in b_test['bidder_id']:
    if i % 100 == 0:
        print(i)
    i += 1
    c = bids[bids.bidder_id == bidder].count()[0]
    b_test.loc[b_test[b_test.bidder_id == bidder].index, 'total_bids'] = c
    # b_test.to_csv('data/b_test.csv', sep=',', header=True, index=False)

In [ ]:
b_train

### 3. total_auctions

In [ ]:
b_train['total_auctions'] = 0
i = 0
for bidder in b_train['bidder_id']:
    if i % 100 == 0:
        print(i)
    i += 1
    c = bids[bids.bidder_id == bidder]['auction'].nunique()
    b_train.loc[b_train[b_train.bidder_id == bidder].index, 'total_auctions'] = c
    # b_train.to_csv('data/b_train.csv', sep=',', header=True, index=False)

In [ ]:
b_test['total_auctions'] = 0
i = 0
for bidder in b_test['bidder_id']:
    if i % 100 == 0:
        print(i)
    i += 1
    c = bids[bids.bidder_id == bidder]['auction'].nunique()
    b_test.loc[b_test[b_test.bidder_id == bidder].index, 'total_auctions'] = c
    # b_test.to_csv('data/b_test.csv', sep=',', header=True, index=False)

In [ ]:
b_train

### 4. bids_per_auction

In [ ]:
b_train['bids_per_auction'] = b_train.total_bids.divide(b_train.total_auctions)
b_train['bids_per_auction'] = round(b_train['bids_per_auction'], 4)

In [ ]:
b_test['bids_per_auction'] = b_test.total_bids.divide(b_test.total_auctions)
b_test['bids_per_auction'] = round(b_test['bids_per_auction'], 4)

In [ ]:
b_train

### 5-6. mean_response & min_response

In [ ]:
# create additional dataset bids_sorted to generate values for feature mean_response & min_response
# already created

bids_sorted = bids.sort_values(by=['auction', 'time'])
del bids_sorted['time']
del bids_sorted['outcome']
bids_sorted['diff'] = bids_sorted.time_normalized - bids_sorted.time_normalized.shift(1)
i = 0
for auc in bids_sorted.auction.unique():
    if i % 200 == 0:
        print(i)
    i += 1
    bids_sorted.loc[bids_sorted[bids_sorted.auction == auc].index, 'diff'] = bids_sorted[bids_sorted.auction == auc].time_normalized - bids_sorted[bids_sorted.auction == auc].time_normalized.shift(1)
    bids_sorted.loc[bids_sorted[bids_sorted.auction == auc].index[0], 'diff'] = np.NaN


In [ ]:
bids_sorted.to_csv('../data/bids_sorted.csv', sep=',', header=True, index=False)

In [ ]:
bids_sorted = pd.read_csv('../data/bids_sorted.csv', sep=',')
print('bids_sorted: OK {}'.format(bids_sorted.shape))
bids_sorted.head()

In [ ]:
b_train['mean_response'] = 0
i = 0
for bidder in b_train['bidder_id'][i:]:
    if i % 100 == 0:
        print(i)
    i += 1
    temp = bids_sorted[bids_sorted.bidder_id == bidder]['diff']
    b_train.loc[b_train[b_train.bidder_id == bidder].index, ['mean_response', 'min_response']] = \
        round(temp.mean(), 4), round(temp.min(), 4)
    # b_train.to_csv('data/b_train.csv', sep=',', header=True, index=False)

In [ ]:
b_test['mean_response'] = 0
i = 0
for bidder in b_test['bidder_id'][i:]:
    if i % 100 == 0:
        print(i)
    i += 1
    temp = bids_sorted[bids_sorted.bidder_id == bidder]['diff']
    b_test.loc[b_test[b_test.bidder_id == bidder].index, ['mean_response', 'min_response']] = \
        round(temp.mean(), 4), round(temp.min(), 4)
    # b_test.to_csv('data/b_test.csv', sep=',', header=True, index=False)

### 7-8. ip_entropy, url_entropy

In [ ]:
def log_entropy(x):
    e = np.sum(np.log(np.array(range(1,np.sum(x)))))
    for i in x:
        e -= np.sum(np.log(np.array(range(1,i))))
    return e

In [ ]:
original_bids = pd.read_csv("../data/bids.csv")
# i=0
# for bidder in b_train['bidder_id'][0:]:
#     if i % 100 == 0:
#         print(i)
#     i+=1

original_bids

In [ ]:
b_train['ip_entropy'] = 0
b_train['url_entropy'] = 0
i = 0
for bidder in b_train['bidder_id'][i:]:
    if i % 100 == 0:
        print(i)
    i += 1
    temp_df = original_bids[bids.bidder_id == bidder][['ip','url']]
    b_train.loc[b_train[b_train.bidder_id == bidder].index, ['ip_entropy', 'url_entropy']] = \
                    round(log_entropy(temp_df.ip.groupby(temp_df.ip).count()),4), \
                    round(log_entropy(temp_df.url.groupby(temp_df.url).count()),4)
    # b_train.to_csv('data/b_train.csv', sep=',', header=True, index=False)

In [ ]:
b_test['ip_entropy'] = 0
b_test['url_entropy'] = 0
i = 0
for bidder in b_test['bidder_id'][i:]:
    if i % 100 == 0:
        print(i)
    i += 1
    temp_df = original_bids[bids.bidder_id == bidder][['ip','url']]
    b_test.loc[b_test[b_test.bidder_id == bidder].index, ['ip_entropy', 'url_entropy']] = \
                    round(log_entropy(temp_df.ip.groupby(temp_df.ip).count()),4), \
                    round(log_entropy(temp_df.url.groupby(temp_df.url).count()),4)
    # b_test.to_csv('data/b_test.csv', sep=',', header=True, index=False)

In [ ]:
df = pd.DataFrame([1,2,1], columns=['h'])
print(log_entropy(df.h.groupby(df.h).count()))
print(df.h.groupby(df.h).count())
df

### look through

In [ ]:
features = ['bidder_id', 'outcome',
         'total_bids', 'total_auctions', 'bids_per_auction',
         'mean_time_diff', 'mean_response', 'min_response',
         'ip_entropy', 'url_entropy']

In [ ]:
b_train.tail()

In [ ]:
b_test.tail()

### save

In [ ]:
b_train.to_csv('../data/b_train.csv', sep=',', header=True, index=False)

In [ ]:
b_test.to_csv('../data/b_test.csv', sep=',', header=True, index=False)